# Compiled Credit LLM, end to end

This notebook runs the whole pipeline on the UCI *Default of Credit Card Clients* benchmark and ends with the assembled
checkpoint answering raw requests. Every stage is one of the repository's scripts; the notebook only orders them and shows
the numbers they produce.

**Where the time goes.** Stages 1 to 7 and 12 to 14 run on the CPU in a few minutes. Stage 9 (teacher), 10 (LoRA) and
11 (gates) need a GPU; on a single mid-range GPU they take roughly an hour, half an hour and a quarter of an hour.
Set `RUN_GPU_STAGES = False` to skip them and use an already built checkpoint for the inference section.

**What the language model does and does not do.** The scorer, the attribution, the policy and the recourse engine are
tensor arithmetic on frozen buffers inside the checkpoint. The language model writes one paragraph about those facts and
the checkpoint verifies the paragraph before releasing it. Nothing the language model does can move the score.

In [1]:
import json, os, subprocess, sys, time
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "compiled_credit_llm").is_dir():
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from compiled_credit_llm import paths

RUN_GPU_STAGES = True          # teacher pass, LoRA training, generation gates
REUSE_EXISTING_TARGETS = True  # skip the teacher pass when data/explanations/dataset_manifest.json already exists
OVERWRITE_OUTPUTS = True       # remove a previous adapter and final checkpoint before rebuilding them
BASE_MODEL = os.environ.get("CCL_BASE_MODEL", "Qwen/Qwen3.5-0.8B")   # Hub id, or a local directory with the official weights
TEACHER = os.environ.get("CCL_TEACHER", "Qwen/Qwen3-4B-Instruct-2507")

def run(script, *args, env=None):
    started = time.time()
    cmd = [sys.executable, str(ROOT / "scripts" / script), *map(str, args)]
    print("$", " ".join(Path(c).name if c.startswith(str(ROOT)) else c for c in cmd))
    proc = subprocess.run(cmd, text=True, capture_output=True, env={**os.environ, **(env or {})})
    out = "\n".join(l for l in (proc.stdout + proc.stderr).splitlines() if "Loading weights" not in l and "clean_up_tokenization" not in l and "falling back" not in l)
    print(out[-6000:])
    print(f"[{time.time() - started:.0f} s, exit {proc.returncode}]")
    if proc.returncode:
        raise RuntimeError(f"{script} failed")

def show(path, keys=None, limit=4000):
    value = json.loads(Path(path).read_text())
    if keys:
        value = {k: value[k] for k in keys if k in value}
    print(json.dumps(value, indent=2)[:limit])

## 1. Data

The repository ships the converted UCI table and the frozen splits, so the download is optional. The split is
client-level and stratified: train 18,000, validation 4,500, calibration 3,000, test 4,500. The test split is read exactly
once, by the scorer's locked evaluation. Nothing on the explanation side ever touches it.

In [2]:
# run("01_download_uci.py")   # optional: re-derive data/default_of_credit_card_clients.csv from the official XLS
run("02_make_splits.py")
show(paths.SPLIT_SUMMARY, ["split_sizes", "overall_prevalence"])

$ python 02_make_splits.py


{
  "train": {
    "rows": 18000,
    "defaults": 3982,
    "non_defaults": 14018,
    "prevalence": 0.2212222222222222
  },
  "validation": {
    "rows": 4500,
    "defaults": 995,
    "non_defaults": 3505,
    "prevalence": 0.22111111111111112
  },
  "calibration": {
    "rows": 3000,
    "defaults": 664,
    "non_defaults": 2336,
    "prevalence": 0.22133333333333333
  },
  "test": {
    "rows": 4500,
    "defaults": 995,
    "non_defaults": 3505,
    "prevalence": 0.22111111111111112
  }
}
[1 s, exit 0]
{
  "split_sizes": {
    "train": 18000,
    "validation": 4500,
    "calibration": 3000,
    "test": 4500
  },
  "overall_prevalence": 0.2212
}


## 2. Scorer

A bounded XGBoost search on the validation split, then a frozen calibration rule (isotonic only if it beats Platt by at
least 0.002 in validation log loss) and the locked test evaluation with bootstrap intervals.

In [3]:
run("03_train_scorer.py")
m = json.loads((paths.SCORER / "metrics.json").read_text())
print("selected trees:", m["selected_trees"], "| calibration:", m["calibration_method"])
for k in ("roc_auc", "ks", "pr_auc", "brier"):
    ci = m["bootstrap"]["intervals"]["uncalibrated"][k]
    print(f"test {k:8s} {m['test_metrics']['uncalibrated'][k]:.4f}  (95% CI {ci['lower_95']:.4f} to {ci['upper_95']:.4f})")

$ python 03_train_scorer.py


{
  "selected_candidate": 5,
  "selected_trees": 113,
  "calibration_method": "platt",
  "test_metrics": {
    "uncalibrated": {
      "roc_auc": 0.777888730385164,
      "ks": 0.43657087147578116,
      "pr_auc": 0.5456351330506021,
      "log_loss": 0.43253411935618313,
      "brier": 0.1358427418424606,
      "calibration_intercept": -0.0318018277795498,
      "calibration_slope": 0.9651971089845942
    },
    "calibrated": {
      "roc_auc": 0.777888730385164,
      "ks": 0.43657087147578116,
      "pr_auc": 0.5456351330506021,
      "log_loss": 0.4326653767021521,
      "brier": 0.1358824438242358,
      "calibration_intercept": -0.07959381241213644,
      "calibration_slope": 0.9567158108610585
    }
  }
}
[10 s, exit 0]
selected trees: 113 | calibration: platt
test roc_auc  0.7779  (95% CI 0.7615 to 0.7947)
test ks       0.4366  (95% CI 0.4079 to 0.4694)
test pr_auc   0.5456  (95% CI 0.5124 to 0.5776)
test brier    0.1358  (95% CI 0.1289 to 0.1425)


## 3. Compile the scorer, the attribution and the policy

Compiling turns the booster into exact fixed-point tensors. Parity is proven on all 30,000 rows: the compiled margin
reproduces the booster's ranking with zero sign inversions and identical ROC-AUC and KS. TreeSHAP is compiled as one
lookup table per tree over the regions induced by the tree's own thresholds; local accuracy is exact in integer units. The
policy is compiled into buffers and audited against an independent NumPy oracle on every row.

In [4]:
run("04_compile_scorer.py")
run("05_compile_treeshap.py")
run("06_compile_policy.py")

$ python 04_compile_scorer.py


{
  "threshold_equivalence": {
    "observed_integer_comparisons": 11952477,
    "observed_integer_mismatches": 0,
    "boundary_integer_comparisons": 6780,
    "boundary_integer_mismatches": 0
  },
  "test": {
    "max_absolute_margin_error": 2.105733032387036e-06,
    "sign_inversions": 0,
    "roc_auc_difference": 0.0,
    "ks_difference": 0.0
  }
}
[4 s, exit 0]
$ python 05_compile_treeshap.py


{
  "regions": 344106,
  "global_max_absolute_error": 8.260782471403161e-07,
  "compiled_local_accuracy_residual": {
    "max_absolute_fixed_units": 0
  }
}
[4 s, exit 0]
$ python 06_compile_policy.py


{
  "policy_state_sha256": "f76c3d47e7ed252b621d40c2551e6338461784e10f53da26068e2f7d8a06ad0d",
  "test_ranking": {
    "margin_auc": 0.777888730385164,
    "probability_auc": 0.777888730385164,
    "margin_ks": 0.43657087147578116,
    "probability_ks": 0.43657087147578116
  },
  "outcomes_test": {
    "INVALID_INPUT": 0,
    "DEMO_DECLINE": 440,
    "MANUAL_REVIEW": 1419,
    "APPROVE_WITH_CONTROLS": 1011,
    "APPROVE_STANDARD": 1630
  }
}
[2 s, exit 0]


## 4. Base checkpoint

The official Qwen3.5-0.8B tensors are copied unchanged (each one is hashed before and after) and the compiled buffers are
added under `compiled_credit.*` in the same safetensors file. The three checkpoint source files are copied beside it.

In [5]:
run("07_build_base_checkpoint.py", "--base", BASE_MODEL)
show(paths.BASE_CHECKPOINT / "checkpoint_receipt.json", ["compiled_state_sha256", "official_tensor_count", "compiled_tensor_count"])

$ python 07_build_base_checkpoint.py --base <local Qwen3.5-0.8B weights>


{
  "compiled_state_sha256": "4d62c3df39b4c25f8d85e9842f530ed3ab57dbd6c5d120a5cc12252aeaca57b6",
  "official_tensor_count": 488,
  "compiled_tensor_count": 3054,
  "packaging_runtime_seconds": 3.06598702800693
}
[6 s, exit 0]
{
  "compiled_state_sha256": "4d62c3df39b4c25f8d85e9842f530ed3ab57dbd6c5d120a5cc12252aeaca57b6",
  "official_tensor_count": 488,
  "compiled_tensor_count": 3054
}


The base checkpoint already answers requests. Without the language model the assessment is complete except for the
paragraph (`explanation_status: NOT_GENERATED`).

In [6]:
import torch
from transformers import AutoModelForImageTextToText, AutoTokenizer
base = AutoModelForImageTextToText.from_pretrained(paths.BASE_CHECKPOINT, trust_remote_code=True).eval()
base_tok = AutoTokenizer.from_pretrained(paths.BASE_CHECKPOINT, trust_remote_code=True)
request = json.loads((ROOT / "examples/requests/decline_close_to_approval.json").read_text())
result = base.assess_credit(request, base_tok, explain=False)
payload = json.loads(result.output)
print(json.dumps({k: payload[k] for k in ("decision", "model", "recourse", "explanation_status")}, indent=2))
print(payload["receipt"])

<venv>/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/3527 [00:00<?, ?it/s]

Loading weights:  48%|████▊     | 1698/3527 [00:00<00:00, 16955.80it/s]

Loading weights:  96%|█████████▌| 3394/3527 [00:00<00:00, 10374.30it/s]

Loading weights: 100%|██████████| 3527/3527 [00:00<00:00, 10457.24it/s]

{
  "decision": {
    "outcome_code": "DD",
    "outcome": "demonstration decline",
    "level": "hard",
    "controlling_rules": [
      "severe delinquency"
    ],
    "note": "first active level in priority order hard > review > soft > approve"
  },
  "model": {
    "probability": "0.698978159399",
    "probability_pct": "69.9%",
    "risk_band": "elevated",
    "raw_margin": 0.842437
  },
  "recourse": {
    "current_outcome": "demonstration decline",
    "current_outcome_code": "DD",
    "scope": "behavioural fields only: repayment statuses, statement balances, payments",
    "non_behavioural_blockers": [],
    "status": "REACHABLE",
    "next_better": {
      "outcome": "approval with controls",
      "outcome_code": "AC",
      "moves_needed": 1,
      "moves": [
        "all six repayment statuses paid on time (no month in delay)"
      ],
      "fields_changed": 5,
      "model_after": {
        "risk_band": "low",
        "probability_pct": "11.7%"
      }
    },
    "best_re

In [7]:
# The seven malformed request classes are refused before anything runs; the language model is never called.
from compiled_credit_llm.data.requests import malformed_mutations
for cls, bad in malformed_mutations(request, result.evidence):
    refused = base.assess_credit(bad, base_tok)
    print(f"{cls:22s} {refused.status}  lm_called={refused.generation_invoked}")
del base

MISSING_FIELD          REFUSE  lm_called=False
DUPLICATE_CONFLICT     REFUSE  lm_called=False
OUT_OF_RANGE           REFUSE  lm_called=False
MALFORMED_TYPE         REFUSE  lm_called=False
EVIDENCE_MISMATCH      REFUSE  lm_called=False
UNSUPPORTED_REQUEST    REFUSE  lm_called=False
OVERRIDE_ATTEMPT       REFUSE  lm_called=False


## 5. Teach the explanation writer

`08_sample_rows.py` chooses development rows and variants that exercise every outcome and every rare rule (validation
rows become the held-back validation, rare-trigger and smoke sets). `09_write_explanations.py` has a teacher model write
one paragraph per row from exactly the facts JSON the checkpoint sends at serving time; the checkpoint's own verifier
decides which paragraphs become targets. Dropped rows simply have no target.

In [8]:
run("08_sample_rows.py")
show(paths.EXPLANATIONS / "rows_summary.json", ["train", "validation", "rare_eval", "smoke"])
if RUN_GPU_STAGES:
    if REUSE_EXISTING_TARGETS and (paths.EXPLANATIONS / "dataset_manifest.json").exists():
        print("reusing the existing explanation targets in", paths.EXPLANATIONS)
    else:
        run("09_write_explanations.py", "--teacher", TEACHER)
    show(paths.EXPLANATIONS / "dataset_manifest.json", ["teacher", "counts", "stats"])

$ python 08_sample_rows.py


{
  "train": {
    "rows": 1390,
    "variants": {
      "N": 800,
      "M": 100,
      "C": 100,
      "O": 100,
      "RU_N": 150,
      "RU_O": 60,
      "DZ_N": 24,
      "DX_N": 56
    },
    "outcomes": {
      "DD": 290,
      "MR": 500,
      "AC": 200,
      "AS": 200,
      "II": 200
    }
  },
  "validation": {
    "rows": 220,
    "variants": {
      "N": 160,
      "M": 20,
      "C": 20,
      "O": 20
    },
    "outcomes": {
      "DD": 40,
      "MR": 60,
      "AC": 40,
      "AS": 40,
      "II": 40
    }
  },
  "rare_eval": {
    "rows": 24,
    "variants": {
      "RU_N": 10,
      "RU_O": 6,
      "DZ_N": 3,
      "DX_N": 5
    },
    "outcomes": {
      "MR": 15,
      "DD": 9
    }
  },
  "smoke": {
    "rows": 5,
    "variants": {
      "M": 1,
      "N": 4
    },
    "outcomes": {
      "II": 1,
      "DD": 1,
      "MR": 1,
      "AC": 1,
      "AS": 1
    }
  }
}

[34 s, exit 0]
{
  "train": {
    "outcomes": {
      "AC": 200,
      "AS": 200,
      "DD": 2

## 6. LoRA, gate, merge

Only the language backbone is adapted (LoRA rank 16); the compiled buffers are hashed before and after. The adapter is
judged by free generation on the real route: raw request in, checkpoint validates, scores, prompts, verifies. The smoke
gate must verify every paragraph and refuse all seven malformed classes with zero language-model calls before the adapter
is merged.

In [9]:
import shutil
if RUN_GPU_STAGES:
    adapter_dir = paths.ADAPTERS / "explainer-lora"
    if OVERWRITE_OUTPUTS:
        for stale in (adapter_dir, paths.FINAL_CHECKPOINT):
            if stale.exists():
                print("removing previous", stale); shutil.rmtree(stale)
    run("10_train_explainer.py")
    adapter = adapter_dir / "final_adapter"
    for partition in ("smoke", "rare_eval", "validation"):
        run("11_gate_explainer.py", "--adapter", adapter, "--partition", partition)
    run("12_merge_checkpoint.py")
    for partition in ("smoke", "rare_eval", "validation"):
        run("11_gate_explainer.py", "--checkpoint", paths.FINAL_CHECKPOINT, "--partition", partition)

removing previous <repo>/artifacts/adapters/explainer-lora
removing previous <repo>/artifacts/checkpoints/CompiledCreditLLM-0.8B-UCI


$ python 10_train_explainer.py


:19<00:17,  5.89it/s]

 50%|█████     | 104/206 [00:19<00:17,  5.87it/s]

 51%|█████     | 105/206 [00:19<00:17,  5.85it/s]

 51%|█████▏    | 106/206 [00:19<00:17,  5.85it/s]

 52%|█████▏    | 107/206 [00:20<00:17,  5.80it/s]

 52%|█████▏    | 108/206 [00:20<00:16,  5.81it/s]

 53%|█████▎    | 109/206 [00:20<00:16,  5.80it/s]

 53%|█████▎    | 110/206 [00:20<00:16,  5.82it/s]

 54%|█████▍    | 111/206 [00:20<00:16,  5.84it/s]

 54%|█████▍    | 112/206 [00:20<00:16,  5.82it/s]

 55%|█████▍    | 113/206 [00:21<00:16,  5.80it/s]

 55%|█████▌    | 114/206 [00:21<00:15,  5.79it/s]

 56%|█████▌    | 115/206 [00:21<00:15,  5.74it/s]

 56%|█████▋    | 116/206 [00:21<00:15,  5.73it/s]

 57%|█████▋    | 117/206 [00:21<00:15,  5.74it/s]

 57%|█████▋    | 118/206 [00:21<00:15,  5.83it/s]

 58%|█████▊    | 119/206 [00:22<00:14,  6.06it/s]

 58%|█████▊    | 120/206 [00:22<00:13,  6.18it/s]

 59%|█████▊    | 121/206 [00:22<00:13,  6.18it/s]

 59%|█████▉    | 122/206 [00:22<00:13,  6.20it/s]

 60%|███

validation-M-client_6d43a1b195cd4eaf785f: VERIFIED
validation-N-client_cb0cb115d5aa5a7beb9a: VERIFIED
validation-N-client_3ef543e5543c5c1f9a4f: VERIFIED
validation-N-client_61284dce6f872d0bf724: VERIFIED
validation-N-client_f9f65a51eeba411ca778: WITHHELD_BY_VERIFIER
{
  "rows": 5,
  "verified": 4,
  "verified_rate": 0.8,
  "violations": {
    "unlicensed numbers": 1
  },
  "gate": "PASS"
} refusals 7 / 7
written <repo>/artifacts/evaluation/gate_adapter_smoke.json

[38 s, exit 0]
$ python 11_gate_explainer.py --adapter final_adapter --partition rare_eval


idation-RU_N-client_0f7d416bbeefbb1edbd7: VERIFIED
idation-RU_N-client_93dd392ccb4126fcfdbf: VERIFIED
idation-RU_N-client_c78a513315f32a1780ab: VERIFIED
idation-RU_N-client_278390d5e8cc651665c4: VERIFIED
idation-RU_N-client_53a4bda781765b908cab: VERIFIED
idation-RU_N-client_b3d2f3246e9ef517503d: VERIFIED
idation-RU_N-client_b2bb6346b3a80323a7f8: VERIFIED
idation-RU_N-client_139a73a34407c89b6844: VERIFIED
idation-RU_N-client_841ebce4840c13460f4f: VERIFIED
idation-RU_O-client_ddfbec08faafe0b4165d: VERIFIED
idation-RU_O-client_ee56d82f42f40465eff7: VERIFIED
idation-RU_O-client_d9b784eae46f7a7533c7: VERIFIED
idation-RU_O-client_f19081de99ed4329c34e: VERIFIED
idation-RU_O-client_a3a162c1180cf07d085e: VERIFIED
idation-DZ_N-client_5833d945596d3bda603e: VERIFIED
idation-DZ_N-client_a0969842591bebfc4365: VERIFIED
idation-DZ_N-client_9682ba85eb32980862e5: VERIFIED
idation-DX_N-client_6d6359b40fcc44734aae: VERIFIED
idation-DX_N-client_ce39825c29e3af65385f: VERIFIED
idation-DX_N-client_5cb68d9e226

98: VERIFIED
validation-N-client_bd8f733129920343eb6b: VERIFIED
validation-N-client_463e4a08ad5f7aa25253: VERIFIED
validation-N-client_a468e3499fd7a979a72a: VERIFIED
validation-N-client_9009a9a131fcfc88af6c: VERIFIED
validation-N-client_9b04f5b2679026885343: VERIFIED
validation-N-client_f70a519973a34c8074d0: VERIFIED
validation-N-client_69217ed50e2eb9fd7b23: VERIFIED
validation-N-client_c16e73d0047e8f8c9916: VERIFIED
validation-N-client_cf22e6d82ea078de2971: VERIFIED
validation-N-client_531b7367ccabe6165139: VERIFIED
validation-N-client_5f8e4a529beae88579bd: VERIFIED
validation-N-client_74c970a77cd0f0b410eb: VERIFIED
validation-N-client_fd775318e0b904dd3afe: VERIFIED
validation-N-client_f2c762fafbb69611db6b: VERIFIED
validation-N-client_0e2f879c48d84a46da32: VERIFIED
validation-N-client_b6d228bd5e6f4b3ec6cc: VERIFIED
validation-N-client_53b9905a40da9cc1002e: VERIFIED
validation-N-client_2e1ddc3a47111ad33c74: VERIFIED
validation-N-client_b25c8a9b09e2dd4406d2: VERIFIED
validation-N-clien

{
  "output": "<repo>/artifacts/checkpoints/CompiledCreditLLM-0.8B-UCI",
  "compiled_state_sha256": "4d62c3df39b4c25f8d85e9842f530ed3ab57dbd6c5d120a5cc12252aeaca57b6",
  "targeted_linear_modules": 186,
  "adapter_sha256": "ff2cfdb096460c46d993b28f66c7b0aefec722da14903234622f1887a8503f67",
  "runtime_seconds": 3.59871067100903
}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.03it/s]
[7 s, exit 0]
$ python 11_gate_explainer.py --checkpoint CompiledCreditLLM-0.8B-UCI --partition smoke


validation-M-client_6d43a1b195cd4eaf785f: VERIFIED
validation-N-client_cb0cb115d5aa5a7beb9a: VERIFIED
validation-N-client_3ef543e5543c5c1f9a4f: VERIFIED
validation-N-client_61284dce6f872d0bf724: VERIFIED
validation-N-client_f9f65a51eeba411ca778: WITHHELD_BY_VERIFIER
{
  "rows": 5,
  "verified": 4,
  "verified_rate": 0.8,
  "violations": {
    "unlicensed numbers": 1
  },
  "gate": "PASS"
} refusals 7 / 7
written <repo>/artifacts/evaluation/gate_CompiledCreditLLM-0.8B-UCI_smoke.json

[30 s, exit 0]
$ python 11_gate_explainer.py --checkpoint CompiledCreditLLM-0.8B-UCI --partition rare_eval


idation-RU_N-client_0f7d416bbeefbb1edbd7: VERIFIED
idation-RU_N-client_93dd392ccb4126fcfdbf: VERIFIED
idation-RU_N-client_c78a513315f32a1780ab: VERIFIED
idation-RU_N-client_278390d5e8cc651665c4: VERIFIED
idation-RU_N-client_53a4bda781765b908cab: VERIFIED
idation-RU_N-client_b3d2f3246e9ef517503d: VERIFIED
idation-RU_N-client_b2bb6346b3a80323a7f8: VERIFIED
idation-RU_N-client_139a73a34407c89b6844: VERIFIED
idation-RU_N-client_841ebce4840c13460f4f: VERIFIED
idation-RU_O-client_ddfbec08faafe0b4165d: VERIFIED
idation-RU_O-client_ee56d82f42f40465eff7: VERIFIED
idation-RU_O-client_d9b784eae46f7a7533c7: VERIFIED
idation-RU_O-client_f19081de99ed4329c34e: VERIFIED
idation-RU_O-client_a3a162c1180cf07d085e: VERIFIED
idation-DZ_N-client_5833d945596d3bda603e: VERIFIED
idation-DZ_N-client_a0969842591bebfc4365: VERIFIED
idation-DZ_N-client_9682ba85eb32980862e5: VERIFIED
idation-DX_N-client_6d6359b40fcc44734aae: VERIFIED
idation-DX_N-client_ce39825c29e3af65385f: VERIFIED
idation-DX_N-client_5cb68d9e226

e5f6fb7ef15c98: VERIFIED
validation-N-client_bd8f733129920343eb6b: VERIFIED
validation-N-client_463e4a08ad5f7aa25253: VERIFIED
validation-N-client_a468e3499fd7a979a72a: VERIFIED
validation-N-client_9009a9a131fcfc88af6c: VERIFIED
validation-N-client_9b04f5b2679026885343: VERIFIED
validation-N-client_f70a519973a34c8074d0: VERIFIED
validation-N-client_69217ed50e2eb9fd7b23: VERIFIED
validation-N-client_c16e73d0047e8f8c9916: VERIFIED
validation-N-client_cf22e6d82ea078de2971: VERIFIED
validation-N-client_531b7367ccabe6165139: VERIFIED
validation-N-client_5f8e4a529beae88579bd: VERIFIED
validation-N-client_74c970a77cd0f0b410eb: VERIFIED
validation-N-client_fd775318e0b904dd3afe: VERIFIED
validation-N-client_f2c762fafbb69611db6b: VERIFIED
validation-N-client_0e2f879c48d84a46da32: VERIFIED
validation-N-client_b6d228bd5e6f4b3ec6cc: VERIFIED
validation-N-client_53b9905a40da9cc1002e: VERIFIED
validation-N-client_2e1ddc3a47111ad33c74: VERIFIED
validation-N-client_b25c8a9b09e2dd4406d2: VERIFIED
valida

## 7. Verify and try to break it

`13_verify_checkpoint.py` reloads the final checkpoint strictly, checks the compiled identity, scores all 30,000 rows
against the base checkpoint (every tensor field exact), checks the policy against the oracle and runs the seven refusals
with the language model forbidden. `14_tamper_determinism.py` changes a tree leaf, a policy bound, the expected hash and
one bit in the file (each must refuse with no language-model call), changes a language weight (scores must not move), and
compares repeats, batch sizes and devices.

In [10]:
run("13_verify_checkpoint.py")
run("14_tamper_determinism.py")
report = json.loads((paths.FINAL_CHECKPOINT / "tamper_determinism_report.json").read_text())
for t in report["tests"]:
    print(f"{t['test']:45s} {'PASS' if t['pass'] else 'FAIL'}")

$ python 13_verify_checkpoint.py


{
  "status": "VERIFIED_CPU_SCOPE",
  "compiled_state_sha256": "4d62c3df39b4c25f8d85e9842f530ed3ab57dbd6c5d120a5cc12252aeaca57b6",
  "tensor_inventory": {
    "total_tensor_count": 3542,
    "compiled_tensor_count": 3054,
    "language_tensor_count": 473,
    "mtp_tensor_count": 15,
    "lora_tensor_count": 0
  },
  "policy_oracle_parity": {
    "rows": 30000,
    "calculations": true,
    "flags": true,
    "risk_band": true,
    "triggers": true,
    "outcome": true
  },
  "deterministic_refusals": {
    "classes_passed": 7,
    "language_model_calls": 0,
    "cases": [
      {
        "class": "MISSING_FIELD",
        "status": "REFUSE",
        "refusal_class": "MISSING_FIELD",
        "exact_text": true,
        "generation_invoked": false
      },
      {
        "class": "DUPLICATE_CONFLICT",
        "status": "REFUSE",
        "refusal_class": "DUPLICATE_CONFLICT",
        "exact_text": true,
        "generation_invoked": false
      },
      {
        "class": "OUT_OF_RANGE",


T1_compiled_leaf_tamper                       PASS
T2_policy_bound_tamper                        PASS
T3_config_hash_tamper                         PASS
T0_restored_after_tamper                      PASS
T4_language_weight_tamper_scores_unchanged    PASS
D1_same_request_twice_cpu                     PASS
D2_batch_invariance                           PASS
T5_file_bit_flip                              PASS
D3_cpu_vs_gpu                                 PASS
all_pass True



[213 s, exit 0]
T1_compiled_leaf_tamper                       PASS
T2_policy_bound_tamper                        PASS
T3_config_hash_tamper                         PASS
T0_restored_after_tamper                      PASS
T4_language_weight_tamper_scores_unchanged    PASS
D1_same_request_twice_cpu                     PASS
D2_batch_invariance                           PASS
T5_file_bit_flip                              PASS
D3_cpu_vs_gpu                                 PASS


## 8. Use it

The final checkpoint answers raw requests with the full assessment and the verified paragraph. The examples below are
real UCI rows from the development splits (never the test split).

In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForImageTextToText.from_pretrained(paths.FINAL_CHECKPOINT, trust_remote_code=True, dtype=torch.bfloat16).to(device).eval()
tokenizer = AutoTokenizer.from_pretrained(paths.FINAL_CHECKPOINT, trust_remote_code=True)
for name in ("decline_close_to_approval", "approval_with_controls", "manual_review_not_behavioural", "invalid_input_missing_field", "malformed_string_value"):
    request = json.loads((ROOT / "examples" / "requests" / f"{name}.json").read_text())
    result = model.assess_credit(request, tokenizer)
    payload = json.loads(result.output)
    print("=" * 100); print(name, "->", result.status)
    if result.status == "OK":
        print(payload["decision"]["outcome"], "|", payload["model"]["probability_pct"], payload["model"]["risk_band"], "| recourse:", payload["recourse"]["status"], payload["recourse"].get("proximity", ""))
        print(payload["explanation_status"], "\n", payload["explanation"])
    else:
        print(payload["refusal_class"], "-", payload["reason"])

Loading weights:   0%|          | 0/3527 [00:00<?, ?it/s]

Loading weights:  43%|████▎     | 1513/3527 [00:00<00:00, 15100.92it/s]

Loading weights:  88%|████████▊ | 3095/3527 [00:00<00:00, 15523.28it/s]

Loading weights: 100%|██████████| 3527/3527 [00:00<00:00, 14947.86it/s]

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


... (2 outputs left out)

decline_close_to_approval -> OK
demonstration decline | 69.9% elevated | recourse: REACHABLE close
VERIFIED 
 The decision is demonstration decline controlled by severe delinquency. The model assigns a 69.9% probability of DEFAULT, placing the risk_band in elevated. Key risk-increasing drivers include most recent repayment status and repayment status three months ago, while risk-decreasing factors are payment amount three months ago and credit limit amount. Calculations show current utilization at 58.93%, max at 62.67%, five delinquent statuses, payment coverage at 3.56%, and a balance trend of -3.44%. A data-quality flag notes a repayment code is undocumented in the UCI source. The case is close to approval with controls, requiring all six repayment statuses to be paid on time with no month in delay. The probability is an association from a benchmark model, not a cause and not a guarantee.


approval_with_controls -> OK
approval with controls | 5.8% low | recourse: REACHABLE close
VERIFIED 
 The decision is approval with controls, controlled by a rising balance trend. The model assigns a 5.8% probability of default, placing the risk_band in low. Key risk-increasing drivers include credit limit amount and statement balance three months ago, while decreasing risk factors are most recent repayment status and payment amount two months ago. The balance trend pct is 54.84, current utilization is 60.9%, and payment coverage is 22.74%, with a data-quality flag noting a repayment code is undocumented in the UCI source. The case is close to standard demonstration approval, requiring only one move: reducing the most recent statement balance to at most 28,848 instead of 48,725 to achieve balance growth below 30% of the limit. The probability is an association from a benchmark model, not a cause and not a guarantee.


manual_review_not_behavioural -> OK
manual review | 10.6% low | recourse: NOT_REACHABLE_BY_BEHAVIOUR 
VERIFIED 
 The decision is manual review because an unknown category requires review and a feature is outside the development range. The model assigns a probability of 10.6% and places the case in the low risk_band. Key risk-increasing drivers are payment amount six months ago and most recent statement balance, while most recent repayment status and most recent payment amount reduce risk. Current utilization is at 1.68%, max at 24.29%, average at 17.06%, with a balance trend of -15.83% and payment dispersion of 9705. Data quality flags include a feature being outside the development range and a demographic category being unknown. The status is not reachable by behaviour due to a demographic category being unknown and a value being outside the development range, both of which cannot be changed by borrower actions. The probability is an association from a benchmark model, not a cause and

invalid_input_missing_field -> OK
invalid input | 84.3% high | recourse: NOT_REACHABLE_BY_BEHAVIOUR 
VERIFIED 
 The decision is invalid input because a required input is missing. The model assigns an 84.3% probability of DEFAULT and places the risk band in high. Key risk-increasing drivers include most recent repayment status and repayment status two months ago, while decreasing risk factors are statement balance three months ago and recorded marital-status category. Calculations show current utilization at 6.0%, max at 20.5%, delinquent statuses at 6, payment coverage at 28.71%, and a balance trend of -14.5%. Data quality flags include a required input is missing, a feature is outside the development range, and a demographic category is unknown. The status is not reachable by behaviour due to blockers including required field missing: recorded sex category, a demographic category is unknown; this cannot be changed by borrower behaviour and routes to human review, and a value is outsid

In [12]:
# The full JSON of one assessment
print(json.dumps(json.loads(model.assess_credit(json.loads((ROOT / "examples/requests/decline_close_to_approval.json").read_text()), tokenizer).output), indent=2))

{
  "protocol": "compiled-credit-llm/assessment-v1",
  "status": "OK",
  "decision": {
    "outcome_code": "DD",
    "outcome": "demonstration decline",
    "level": "hard",
    "controlling_rules": [
      "severe delinquency"
    ],
    "note": "first active level in priority order hard > review > soft > approve"
  },
  "model": {
    "probability": "0.698978159399",
    "probability_pct": "69.9%",
    "risk_band": "elevated",
    "raw_margin": 0.842437
  },
  "policy": {
    "id": "UCI_DEMO_POLICY_v1",
    "priority": [
      "hard",
      "review",
      "soft",
      "approve"
    ],
    "active": {
      "hard": [
        "severe delinquency"
      ],
      "review": [
        "multiple soft controls require review"
      ],
      "soft": [
        "repeated delinquency",
        "low payment coverage",
        "elevated model risk"
      ]
    }
  },
  "drivers": {
    "increasing_risk": [
      {
        "feature": "Most recent repayment status",
        "code": "PAY_0",
      

## Boundaries

The policy is a synthetic demonstration policy on a 2005 benchmark; the probability is a benchmark output; TreeSHAP
contributions are associations in a frozen model; recourse describes what the compiled model and the policy would do with
different behavioural inputs. The paragraph is the only free text and it is verified, not trusted. See `docs/` for the
architecture, the policy, the output schema, the evaluation and how to adapt the method to another portfolio.